# 02. ROI-cropped BCE와 Dice Loss

## 목표
Mask logit과 soft target에서 BCE·Dice loss를 계산하고, ground-truth box 안으로 crop할 때 background가 loss에 미치는 영향이 어떻게 달라지는지 확인합니다.

In [ ]:
from math import exp, log

logits = [
    [-2.0, -1.0, -1.0, -2.0],
    [-2.0,  2.0,  1.0, -2.0],
    [-2.0,  1.5, 0.5, -2.0],
    [-2.0, -1.0, -1.0, -2.0],
]
targets = [
    [0.0, 0.0, 0.0, 0.0],
    [0.0, 1.0, 1.0, 0.0],
    [0.0, 1.0, 0.6, 0.0],  # bilinear resize 후 soft target을 흉내 냅니다.
    [0.0, 0.0, 0.0, 0.0],
]
roi = (1, 1, 3, 3)  # x1, y1, x2, y2

def sigmoid(value):
    return 1.0 / (1.0 + exp(-value))

In [ ]:
def select_pixels(matrix, crop=None):
    if crop is None:
        return [value for row in matrix for value in row]
    x1, y1, x2, y2 = crop
    return [matrix[y][x] for y in range(y1, y2) for x in range(x1, x2)]

def bce_loss(mask_logits, mask_targets, crop=None):
    probabilities = [sigmoid(value) for value in select_pixels(mask_logits, crop)]
    selected_targets = select_pixels(mask_targets, crop)
    losses = []
    for probability, target in zip(probabilities, selected_targets):
        probability = min(max(probability, 1e-7), 1 - 1e-7)
        losses.append(-(target * log(probability) + (1 - target) * log(1 - probability)))
    return sum(losses) / len(losses)

def dice_loss(mask_logits, mask_targets, crop=None, epsilon=1e-6):
    probabilities = [sigmoid(value) for value in select_pixels(mask_logits, crop)]
    selected_targets = select_pixels(mask_targets, crop)
    numerator = 2 * sum(p * t for p, t in zip(probabilities, selected_targets)) + epsilon
    denominator = sum(probabilities) + sum(selected_targets) + epsilon
    return 1 - numerator / denominator

for crop in (None, roi):
    name = "full map" if crop is None else "ROI crop"
    print(name, f"BCE={bce_loss(logits, targets, crop):.4f}", f"Dice={dice_loss(logits, targets, crop):.4f}")

## 해석과 확장

ROI crop은 넓은 background pixel이 평균 loss를 지배하지 않게 합니다. 반면 box 밖 mask 오류는 직접 supervision하지 않으므로 postprocessing에서 box 밖 pixel을 제거합니다.

1. ROI를 넓히거나 줄여 loss 변화를 확인하세요.
2. 작은 객체와 큰 객체가 instance 평균에서 같은 비중을 갖는 이유를 설명하세요.
3. Matching cost는 full map, training loss는 ROI라는 차이를 다음 노트북과 연결하세요.